# Notebook 6: Mapa georreferenciado — severidad y estrés hídrico por comuna

Toma la tabla agregada comuna × año que produjo Spark (`04_agregacion_spark_comuna_anio.ipynb`)
y la pinta sobre un mapa real de Chile (polígonos de comuna, no puntos) — coroplético
interactivo para audiencia no técnica.

**Fuente de los límites geográficos:** no había ningún shapefile/GeoJSON de comunas en la
carpeta del proyecto, así que se usa un GeoJSON público y verificado —
[`fcortes/Chile-GeoJSON`](https://github.com/fcortes/Chile-GeoJSON) (346 comunas, campos
`Comuna`/`Region`/`Provincia`, WGS-84). Se descarga directo desde GitHub en la celda 5 — no
hay que subir nada nuevo a Drive.

Produce dos mapas:
1. **Severidad histórica acumulada (2010-2019)** — superficie quemada total por comuna.
2. **Estrés hídrico promedio** — `estres_hidrico_score` promedio por comuna (de la sección 8
   del notebook 04: temperatura anómala alta − precipitación anómala baja).


## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Instalar librerías

In [ ]:
!pip install -q folium
print('✓ Librerías instaladas')


## 3. Configuración — EDITA la ruta antes de correr

In [ ]:
import os, glob

# ⚠️ Debe ser la MISMA carpeta base usada en los notebooks anteriores
DRIVE_BASE = '/content/drive/MyDrive/CAMBIAR_A_TU_RUTA'

SPARK_OUTPUT_DIR = f'{DRIVE_BASE}/datos_procesados/spark_comuna_anio'
OUTPUT_DIR = f'{DRIVE_BASE}/datos_procesados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

GEOJSON_URL = 'https://raw.githubusercontent.com/fcortes/Chile-GeoJSON/master/comunas.geojson'

print('Entrada Spark :', SPARK_OUTPUT_DIR)
print('GeoJSON       :', GEOJSON_URL)


## 4. Cargar la tabla comuna × año generada por Spark

Spark la escribió con `coalesce(1).write.csv(...)`, lo que crea una **carpeta** con un
archivo `part-*.csv` adentro (más un `_SUCCESS`) — hay que buscarlo con `glob`, no es un
archivo con nombre fijo.


In [ ]:
import pandas as pd

csv_matches = glob.glob(f'{SPARK_OUTPUT_DIR}/csv/part-*.csv')
assert csv_matches, f'No se encontró ningún part-*.csv en {SPARK_OUTPUT_DIR}/csv — ¿corriste el notebook 04?'

agg = pd.read_csv(csv_matches[0])
print(f'✓ {agg.shape[0]} filas comuna-año cargadas · {agg.shape[1]} columnas')
agg.head()


## 5. Descargar el GeoJSON de comunas de Chile

Se carga como diccionario Python (no hace falta `geopandas`/`shapely` para esto — `folium`
acepta el GeoJSON crudo directamente).


In [ ]:
import requests

geojson_comunas = requests.get(GEOJSON_URL, timeout=30).json()
print(f'✓ GeoJSON descargado: {len(geojson_comunas["features"])} features (comunas)')
print('Ejemplo de propiedades de una comuna:', geojson_comunas['features'][0]['properties'])


## 6. Normalizar nombres de comuna para el cruce

Los nombres de comuna en el CSV de CONAF y en el GeoJSON pueden diferir en tildes/mayúsculas
(ej. "Aysén" vs "AYSEN"). Se normaliza (mayúsculas, sin tildes) en ambos lados antes de unir
— y se reporta cuántas comunas NO logran match, para no perder datos en silencio.


In [ ]:
import unicodedata

def normalizar(texto):
    if pd.isna(texto):
        return None
    texto = str(texto).strip().upper()
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')

agg['comuna_norm'] = agg['comuna'].apply(normalizar)

for feature in geojson_comunas['features']:
    feature['properties']['comuna_norm'] = normalizar(feature['properties'].get('Comuna'))

nombres_geojson = {f['properties']['comuna_norm'] for f in geojson_comunas['features']}
nombres_datos = set(agg['comuna_norm'].dropna().unique())

sin_match = nombres_datos - nombres_geojson
print(f'Comunas en los datos: {len(nombres_datos)}')
print(f'Comunas SIN match en el GeoJSON: {len(sin_match)}')
if sin_match:
    print('  ->', sorted(sin_match))

superficie_total = agg['superficie_total_ha'].sum()
superficie_sin_match = agg.loc[agg['comuna_norm'].isin(sin_match), 'superficie_total_ha'].sum()
print(f'\n% de superficie_total_ha cubierta por comunas CON match: '
      f'{100 * (1 - superficie_sin_match / superficie_total):.1f}%')


## 7. Agregar por comuna (todo el período 2010-2019)

Una fila = una comuna, resumiendo los 10 años: severidad acumulada, incendios totales y
estrés hídrico promedio — la vista que se pinta en el mapa.


In [ ]:
por_comuna = (
    agg.groupby(['comuna_norm', 'comuna', 'region'], as_index=False)
    .agg(
        n_incendios_total=('n_incendios', 'sum'),
        superficie_total_ha=('superficie_total_ha', 'sum'),
        n_catastroficos_total=('n_catastroficos', 'sum'),
        estres_hidrico_promedio=('estres_hidrico_score', 'mean'),
    )
)

print(f'✓ {len(por_comuna)} comunas agregadas para el mapa')
por_comuna.sort_values('superficie_total_ha', ascending=False).head(10)


## 8. Mapa 1 — Severidad, con filtro por año

Una capa "Todo el período (2010-2019)" (activa por defecto) + una capa por año — se activan
desde el control de capas (arriba a la derecha). **Selecciona un solo año a la vez**: si
activas dos, se dibujan superpuestas y solo se ve la de más arriba.

Escala de color por **cuantiles** (no lineal) y **una sola leyenda** — la superficie quemada
por comuna está muy sesgada (pocas comunas concentran la mayor parte del total), así que una
escala lineal de 0 al máximo deja casi todo el mapa en el color más bajo y solo la comuna más
extrema se distingue. Con cuantiles, cada color representa ~20% de las comunas según su propio
ranking, y se ve variación real.

**Nota:** las capas por año usan esta MISMA escala (calculada sobre el acumulado 2010-2019) a
propósito, para que los colores sean comparables entre año y período completo — por lo mismo,
es normal y esperado que un año individual se vea con tonos más bajos que el acumulado (es una
fracción de 10 años de superficie, no un error).


In [ ]:
import folium
import branca.colormap as bcm
import copy
import numpy as np

anios_disponibles = sorted(int(a) for a in agg['año'].dropna().unique())

m1 = folium.Map(location=[-38.5, -71.5], zoom_start=5, tiles='CartoDB positron')

# Colorear por PERCENTIL relativo (0-100) en vez de por valor absoluto: con datos tan sesgados
# (1-2 comunas concentran la mayor parte de la superficie quemada), una escala por valor
# absoluto queda estirada por ese outlier y aplasta a todas las demás comunas cerca de un
# extremo. El percentil siempre se distribuye parejo 0-100 sin importar qué tan extremo sea
# el outlier. La referencia (a qué distribución se compara) es siempre el acumulado 2010-2019,
# para que el filtro por año sea comparable contra el mismo ranking.
referencia_severidad = np.sort(por_comuna['superficie_total_ha'].values)

def a_percentil_severidad(v):
    return 100 * np.searchsorted(referencia_severidad, v, side='right') / len(referencia_severidad)

colormap_severidad = bcm.LinearColormap(
    colors=['#ffffb2', '#fed976', '#fd8d3c', '#f03b20', '#bd0026'],
    vmin=0, vmax=100,
    caption='Percentil de superficie quemada (relativo a comunas 2010-2019; el valor real está en el tooltip)',
)

def capa_severidad(df_sub, nombre, activa):
    valores = dict(zip(df_sub['comuna_norm'], df_sub['superficie_total_ha']))

    # Copia propia del GeoJSON por capa: se inyecta el valor real como propiedad para que el
    # tooltip lo muestre (geojson_comunas es un solo objeto compartido entre TODAS las capas —
    # mutarlo directo pisaría el valor de una capa con el de la siguiente).
    gj = copy.deepcopy(geojson_comunas)
    for feature in gj['features']:
        v = valores.get(feature['properties']['comuna_norm'])
        feature['properties']['valor_mostrado'] = f'{v:,.0f} ha' if v is not None else 'Sin datos'

    def estilo(feature):
        v = valores.get(feature['properties']['comuna_norm'])
        if v is None:
            return {'fillColor': '#d9d9d9', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
        return {'fillColor': colormap_severidad(a_percentil_severidad(v)), 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.85}

    return folium.GeoJson(
        gj, name=nombre, show=activa, style_function=estilo,
        tooltip=folium.GeoJsonTooltip(
            fields=['Comuna', 'Region', 'valor_mostrado'],
            aliases=['Comuna:', 'Región:', 'Superficie quemada:'],
        ),
    )

capa_severidad(por_comuna, 'Todo el período (2010-2019)', True).add_to(m1)
for anio in anios_disponibles:
    capa_severidad(agg[agg['año'] == anio], f'Año {anio}', False).add_to(m1)

colormap_severidad.add_to(m1)
folium.LayerControl(collapsed=False).add_to(m1)

out_mapa1 = f'{OUTPUT_DIR}/mapa_severidad_historica.html'
m1.save(out_mapa1)
print(f'✓ Mapa guardado: {out_mapa1} · {len(anios_disponibles)} capas de año + 1 acumulada')
m1


## 9. Mapa 2 — Estrés hídrico, con filtro por año

Mismo patrón que el mapa 1: capa "Todo el período" + una capa por año, un solo control de
capas. Escala por **percentil relativo** (0-100), no por valor absoluto — igual que el mapa 1,
1-2 comunas con un score extremo estiraban la escala y aplastaban a todas las demás cerca de
cero (por eso se veía "todo lejos"). 50 = normal/mediana, 100 = más caliente/seco relativo,
0 = más frío/húmedo relativo. El valor real (score con signo) sigue disponible en el tooltip.


In [ ]:
import folium
import branca.colormap as bcm
import copy
import numpy as np

anios_disponibles = sorted(int(a) for a in agg['año'].dropna().unique())

m2 = folium.Map(location=[-38.5, -71.5], zoom_start=5, tiles='CartoDB positron')

referencia_estres = np.sort(por_comuna['estres_hidrico_promedio'].values)

def a_percentil_estres(v):
    return 100 * np.searchsorted(referencia_estres, v, side='right') / len(referencia_estres)

colormap_estres = bcm.LinearColormap(
    colors=['#2166ac', '#ffffbf', '#b2182b'],  # azul (frío/húmedo) -> amarillo pálido (normal) -> rojo (caliente/seco)
    vmin=0, vmax=100,
    caption='Percentil de estrés hídrico (50 = normal; el score real está en el tooltip)',
)

def capa_estres(df_sub, columna_valor, nombre, activa):
    valores = dict(zip(df_sub['comuna_norm'], df_sub[columna_valor]))

    gj = copy.deepcopy(geojson_comunas)
    for feature in gj['features']:
        v = valores.get(feature['properties']['comuna_norm'])
        feature['properties']['valor_mostrado'] = f'{v:+.2f}' if v is not None else 'Sin datos'

    def estilo(feature):
        v = valores.get(feature['properties']['comuna_norm'])
        if v is None:
            return {'fillColor': '#d9d9d9', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
        return {'fillColor': colormap_estres(a_percentil_estres(v)), 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.85}

    return folium.GeoJson(
        gj, name=nombre, show=activa, style_function=estilo,
        tooltip=folium.GeoJsonTooltip(
            fields=['Comuna', 'Region', 'valor_mostrado'],
            aliases=['Comuna:', 'Región:', 'Estrés hídrico:'],
        ),
    )

capa_estres(por_comuna, 'estres_hidrico_promedio', 'Todo el período (2010-2019)', True).add_to(m2)
for anio in anios_disponibles:
    capa_estres(agg[agg['año'] == anio], 'estres_hidrico_score', f'Año {anio}', False).add_to(m2)

colormap_estres.add_to(m2)
folium.LayerControl(collapsed=False).add_to(m2)

out_mapa2 = f'{OUTPUT_DIR}/mapa_estres_hidrico.html'
m2.save(out_mapa2)
print(f'✓ Mapa guardado: {out_mapa2} · {len(anios_disponibles)} capas de año + 1 acumulada')
m2


## ✅ Checklist de validación

- [x] Se reporta explícitamente cuántas comunas no encontraron match en el GeoJSON (sección 6)
      y qué % de la superficie quemada total representan — si es alto (>10-15%), revisar
      manualmente esos nombres antes de la presentación.
- [x] Los dos mapas son HTML autocontenidos — se pueden abrir directo en un navegador o
      incrustar en el dashboard/informe sin depender de Colab.
- [x] Filtro por año (sección 8 y 9): una capa por año + "Todo el período", con escala de
      color fija y una sola leyenda — comparable entre años, no solo un acumulado estático.

**Nota (mejora opcional si sobra tiempo):** el filtro actual es por capas (checkbox, un año
a la vez recomendado). Un slider real (`folium.plugins.TimeSliderChoropleth`, arrastrar para
recorrer los años) se ve más pulido pero es trabajo adicional no crítico — mencionarlo como
"mejora futura" en la presentación es razonable si no alcanza el tiempo.
